## Type I - III diffusable nodes for 3954 topology using Latin Hypercube Sampling (LHS)

**Diffusion Rates:**

**Type I**   
DU, DV, DW 

**Type II**  
DU, DV, DW 

**Type III**  
DU, DV, DW

Then for the LHS/robustness comparison, scan d from 0.1 to 10 and measure what fraction of our stable samples still gives Turing instability at each d value.

We should see Type I collapse to zero as d → 1, while Type II and III stay robustly non-zero...

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from numpy.linalg import eigvals
from scipy.optimize import fsolve
from scipy.stats import qmc

In [2]:
# hill coefficient and hill functions

n = 2

def hill_activation(X, K):
    return X**n / (K**n + X**n)

def hill_inhibition(X, K):
    return K**n / (K**n + X**n)

def dH_act(x, K):
    return n * K**n * x**(n-1) / (K**n + x**n)**2

def dH_inh(x, K):
    return -n * K**n * x**(n-1) / (K**n + x**n)**2

In [3]:
def ode_system(state, params):
    u, v, w = state
    alpha_u, beta_u, K_uu, K_vu, delta_u = params[0:5]
    alpha_v, beta_v, K_uv, K_wv, delta_v = params[5:10]
    alpha_w, beta_w, K_ww, K_uw, K_vw, delta_w = params[10:16]

    du = alpha_u + beta_u * hill_activation(u, K_uu) * hill_inhibition(v, K_vu) - delta_u * u
    dv = alpha_v + beta_v * hill_activation(u, K_uv) * hill_inhibition(w, K_wv) - delta_v * v
    dw = alpha_w + beta_w * hill_activation(w, K_ww) * hill_inhibition(u, K_uw) * hill_inhibition(v, K_vw) - delta_w * w

    return [du, dv, dw]

In [4]:
def find_steady_state(params, n_attempts=10):
    for _ in range(n_attempts):
        initial_guess = np.random.uniform(0.01, 10.0, 3)
        sol = fsolve(ode_system, initial_guess, args=(params,), full_output=True)
        steady_state, info, ier, msg = sol
        residuals = ode_system(steady_state, params)
        if (ier == 1 and
            np.max(np.abs(residuals)) < 1e-8 and
            np.all(steady_state > 0)):
            return steady_state
    return None

In [5]:
def compute_jacobian(state, params):
    u, v, w = state
    alpha_u, beta_u, K_uu, K_vu, delta_u = params[0:5]
    alpha_v, beta_v, K_uv, K_wv, delta_v = params[5:10]
    alpha_w, beta_w, K_ww, K_uw, K_vw, delta_w = params[10:16]

    J = np.zeros((3, 3))

    # Row 0: d(du)/d(u,v,w)
    J[0, 0] = beta_u * dH_act(u, K_uu) * hill_inhibition(v, K_vu) - delta_u
    J[0, 1] = beta_u * hill_activation(u, K_uu) * dH_inh(v, K_vu)
    J[0, 2] = 0

    # Row 1: d(dv)/d(u,v,w)
    J[1, 0] = beta_v * dH_act(u, K_uv) * hill_inhibition(w, K_wv)
    J[1, 1] = -delta_v
    J[1, 2] = beta_v * hill_activation(u, K_uv) * dH_inh(w, K_wv)

    # Row 2: d(dw)/d(u,v,w)
    J[2, 0] = beta_w * hill_activation(w, K_ww) * dH_inh(u, K_uw) * hill_inhibition(v, K_vw)
    J[2, 1] = beta_w * hill_activation(w, K_ww) * hill_inhibition(u, K_uw) * dH_inh(v, K_vw)
    J[2, 2] = beta_w * dH_act(w, K_ww) * hill_inhibition(u, K_uw) * hill_inhibition(v, K_vw) - delta_w

    return J

In [6]:
def is_turing_diego(J, DU, DV, DW):
    # STEP 1: Check stability at k=0 using Routh-Hurwitz
    a1_0 = -np.trace(J)
    a2_0 = (J[0,0]*J[1,1] - J[0,1]*J[1,0] +
            J[0,0]*J[2,2] - J[0,2]*J[2,0] +
            J[1,1]*J[2,2] - J[1,2]*J[2,1])
    a3_0 = -np.linalg.det(J)
    
    # Stable without diffusion: a1>0, a3>0, a1*a2-a3>0
    if not (a1_0 > 0 and a3_0 > 0 and a1_0*a2_0 - a3_0 > 0):
        return False
    
    # STEP 2: Check for stationary instability with diffusion
    # Diego Eq. 3: a₁(q)>0, a₂(q)>0, a₃(q)<0
    D = np.diag([DU, DV, DW])
    
    for k in np.linspace(0.1, 10, 100):
        M = J - k**2 * D
        a1 = -np.trace(M)
        a2 = (M[0,0]*M[1,1] - M[0,1]*M[1,0] +
              M[0,0]*M[2,2] - M[0,2]*M[2,0] +
              M[1,1]*M[2,2] - M[1,2]*M[2,1])
        a3 = -np.linalg.det(M)
        
        # Diego Eq. 3: EXACT condition from paper
        if a1 > 0 and a2 > 0 and a3 < 0:
            return True
    
    return False

# def is_stable(J):
#     return np.all(np.real(eigvals(J)) < 0)

# def is_turing_diego(J, DU, DV, DW):
#     D = np.diag([DU, DV, DW])
#     for k in np.logspace(-1, 2, 100):          # 500 to 100 bc otherwise long runtimes, change to 500 once we use HPC
#         M  = J - k**2 * D
#         a1 = -np.trace(M)
#         a2 = (M[0,0]*M[1,1] - M[0,1]*M[1,0] +
#               M[0,0]*M[2,2] - M[0,2]*M[2,0] +
#               M[1,1]*M[2,2] - M[1,2]*M[2,1])
#         a3 = -np.linalg.det(M)
#         if a3 < 0 and a1 > 0 and a2 > 0:
#             return True
#     return False


In [7]:
def is_turing_shaberi(J, eigs_0, DU, DV, DW):
    # STEP 1: Check stability at k=0 (without diffusion), (Shaberi page 3: "Re(λmax(k=0)) < 0")
    if np.max(np.real(eigs_0)) >= 0:
        return None
    
    # STEP 2: Check for instability with diffusion, (Shaberi page 10: "k from 0 to kmax=10, step size Δk=0.01")
    D = np.diag([DU, DV, DW])
    k_values = np.arange(0.01, 10.01, 0.01)
    
    has_instability = False
    is_oscillatory = False

    for k in k_values:
        M = J - k**2 * D
        eigs_k = np.linalg.eigvals(M)
    
        # Shaberi page 3: "Re(λmax(k)) > 0"
        if np.max(np.real(eigs_k)) > 0:
            has_instability = True
            
            # Check if Turing-Hopf: unstable eigenvalue is complex
            unstable_eigs = eigs_k[np.real(eigs_k) > 0]
            if np.any(np.abs(np.imag(unstable_eigs)) > 1e-8):
                is_oscillatory = True
                break
    
    if not has_instability:
        return None
    
    if is_oscillatory:
        return 'Hopf'

    # STEP 3: Type I vs Type II classification
    k_high_values = np.linspace(10, 50, 20)  # Check k=10 to k=50
    for k in k_high_values:
        M = J - k**2 * D
        eigs_k = np.linalg.eigvals(M)
        if np.max(np.real(eigs_k)) < 0:
            return 'Type-I'  # Restabilizes at high k
        
    return 'Type-II'  # Stays unstable at high k

# def is_turing_shaberi(J, DU, DV, DW):
#     D = np.diag([DU, DV, DW])
#     for k in np.linspace(0.1, 10, 100):
#         M = J - k**2 * D
#         if np.max(np.real(eigvals(M))) > 0:
#             return True
#     return False

# def is_turing_shaberi(J, DU, DV, DW):
#     # FIRST: Check stability at k=0 (without diffusion)
#     eigs_0 = eigvals(J)
#     if np.max(np.real(eigs_0)) >= 0:
#         return False  # Unstable without diffusion - not Turing
#     # SECOND: Check for instability with diffusion
#     D = np.diag([DU, DV, DW])
#     for k in np.linspace(0.1, 10, 100):
#         M = J - k**2 * D
#         eigs_k = eigvals(M)
#         if np.max(np.real(eigs_k)) > 0:
#             return True  # Turing instability found!
#     return False


In [9]:
param_ranges = [
    (0.001, 0.1),  # alpha_U
    (0.1, 10),     # beta_U
    (0.01, 1),     # K_UU  
    (0.01, 1),     # K_VU 
    (0.01, 1),     # delta_U
    (0.001, 0.1),  # alpha_V
    (0.1, 10),     # beta_V
    (0.01, 1),     # K_UV
    (0.01, 1),     # K_WV
    (0.01, 1),     # delta_V
    (0.001, 0.1),  # alpha_W
    (0.1, 10),     # beta_W
    (0.01, 1),     # K_WW
    (0.01, 1),     # K_UW
    (0.01, 1),     # K_VW
    (0.01, 1),     # delta_W
]

n_samples = 10_000
DU, DV, DW = 0.0, 0.01, 0.1

sampler = qmc.LatinHypercube(d=16, seed=42)   
samples = sampler.random(n=n_samples)
params_log = np.zeros((n_samples, 16))  

for i in range(16):  
    log_min = np.log10(param_ranges[i][0])
    log_max = np.log10(param_ranges[i][1])
    params_log[:, i] = 10**(log_min + samples[:, i] * (log_max - log_min))

In [10]:
# Initialize counters
steady_states = 0
stable_without_diffusion = 0
diego_turing = 0 # stationary only: Type I + Type II combined
shaberi_total = 0
shaberi_type_I = 0
shaberi_type_II = 0
shaberi_hopf = 0

np.random.seed(42)
for i in range(n_samples):
    params = params_log[i]
    steady = find_steady_state(params)
    
    if steady is not None:
        steady_states += 1
        J = compute_jacobian(steady, params)
        
        # Compute eigenvalues, check stability at k=0
        eigs_0 = np.linalg.eigvals(J)
        if np.max(np.real(eigs_0)) < 0:
            stable_without_diffusion += 1
            
            # Diego (uses trace/det, detects stationary only)
            if is_turing_diego(J, DU, DV, DW):
                diego_turing += 1
            
            # Shaberi (reuses eigs_0, detects and classifies all types)
            turing_type = is_turing_shaberi(J, eigs_0, DU, DV, DW)
            
            if turing_type is not None:
                shaberi_total += 1
                if turing_type == 'Type-I':
                    shaberi_type_I += 1
                elif turing_type == 'Type-II':
                    shaberi_type_II += 1
                elif turing_type == 'Hopf':
                    shaberi_hopf += 1

    # Progress tracking (for HPC)
    if (i+1) % 100000 == 0:
        print(f"{i+1:,} | Stable: {stable_without_diffusion} | "
              f"Diego: {diego_turing} | Shaberi: {shaberi_total} "
              f"(I:{shaberi_type_I}, II:{shaberi_type_II}, Hopf:{shaberi_hopf})")
              
# Calculate robustness scores
rob_diego = 100 * diego_turing / stable_without_diffusion if stable_without_diffusion > 0 else 0.0
rob_shaberi_total = 100 * shaberi_total / stable_without_diffusion if stable_without_diffusion > 0 else 0.0
rob_shaberi_type_I = 100 * shaberi_type_I / stable_without_diffusion if stable_without_diffusion > 0 else 0.0
rob_shaberi_excl_II = 100 * (shaberi_type_I + shaberi_hopf) / stable_without_diffusion if stable_without_diffusion > 0 else 0.0

# Save results in dictionary
results_3954_lhs = {
    "diffusion_type1": f"U={DU}, V={DV}, W={DW}",
    "n_samples": n_samples,
    "steady_states": steady_states,
    "stable_without_diffusion": stable_without_diffusion,
    "diego_turing": diego_turing,
    "shaberi_total": shaberi_total,
    "shaberi_type_I": shaberi_type_I,
    "shaberi_type_II": shaberi_type_II,
    "shaberi_hopf": shaberi_hopf,
    "rob_diego": rob_diego,
    "rob_shaberi_total": rob_shaberi_total,
    "rob_shaberi_type_I": rob_shaberi_type_I,
    "rob_shaberi_excl_II": rob_shaberi_excl_II
}

# Print as table
print(f"{'Diffusion':<15} {'Tested':<10} {'Steady':<10} {'Stable':<10} "
      f"{'Diego_Tu':<10} {'Shab_Total':<12} {'Shab_I':<10} {'Shab_II':<10} {'Shab_Hopf':<12} "
      f"{'Diego_Rob%':<12} {'Shab_Tot%':<12} {'Shab_I%':<12}")
print("-" * 155)
print(f"{results_3954_lhs['n_samples']:<10,} "
      f"{results_3954_lhs['steady_states']:<10,} "
      f"{results_3954_lhs['stable_without_diffusion']:<10,} "
      f"{results_3954_lhs['diego_turing']:<10} "
      f"{results_3954_lhs['shaberi_total']:<12} "
      f"{results_3954_lhs['shaberi_type_I']:<10} "
      f"{results_3954_lhs['shaberi_type_II']:<10} "
      f"{results_3954_lhs['shaberi_hopf']:<12} "
      f"{results_3954_lhs['rob_diego']:<12.6f} "
      f"{results_3954_lhs['rob_shaberi_total']:<12.6f} "
      f"{results_3954_lhs['rob_shaberi_type_I']:<12.6f}"
      f"{results_3954_lhs['diffusion_type1']:<15} ")

Diffusion       Tested     Steady     Stable     Diego_Tu   Shab_Total   Shab_I     Shab_II    Shab_Hopf    Diego_Rob%   Shab_Tot%    Shab_I%     
-----------------------------------------------------------------------------------------------------------------------------------------------------------
10,000     9,696      9,495      28         28           0          28         0            0.294892     0.294892     0.000000    U=0.0, V=0.01, W=0.1 


In [ ]:
# steady_found   = 0
# stable_count   = 0
# turing_diego   = 0
# turing_shaberi = 0

# np.random.seed(42)
# for i in range(n_samples):
#     params = params_log[i]
#     steady = find_steady_state(params)

#     if steady is not None:
#         steady_found += 1
#         J = compute_jacobian(steady, params)

#         if is_stable(J):
#             stable_count += 1
#             if is_turing_diego(J, DU, DV, DW):
#                 turing_diego += 1
#             if is_turing_shaberi(J, DU, DV, DW):
#                 turing_shaberi += 1

# # print results
# rob_diego   = 100 * turing_diego   / stable_count if stable_count > 0 else 0.0
# rob_shaberi = 100 * turing_shaberi / stable_count if stable_count > 0 else 0.0

# results_3954_lhs = {
#     "n_samples":      n_samples,
#     "steady_found":   steady_found,
#     "stable":         stable_count,
#     "diego":          turing_diego,
#     "shaberi":        turing_shaberi,
#     "rob_diego":      rob_diego,
#     "rob_shaberi":    rob_shaberi,
# }

# print(f"{'Type':<10} {'Tested':<10} {'Steady':<10} {'Stable':<10} {'Diego_Tu':<10} {'Shaberi_Tu':<12} {'Diego_Ro':<12} {'Shaberi_Ro':<12}")
# print("-" * 95)

# print(f"{'Type X':<10} "
#       f"{results_3954_lhs['n_samples']:<10,} "
#       f"{results_3954_lhs['steady_found']:<10,} "
#       f"{results_3954_lhs['stable']:<10,} "
#       f"{results_3954_lhs['diego']:<10,} "
#       f"{results_3954_lhs['shaberi']:<10,} "
#       f"{results_3954_lhs['rob_diego']:>11.7f}% "
#       f"{results_3954_lhs['rob_shaberi']:>11.7f}%")